# DATA MINING EN ECONOMIA Y FINANZAS. 2026
  Comisión jueves

# Tarea para el Hogar 02



Esta Tarea para el Hogar 02 llega a sus manos luego de la segunda clase.
<br> Se espera de usted que intente avanzar con los desafios propuestos y que los traiga terminados para la Clase 03, ya que se analizarán los resultados.

##  1. Ensembles de Modelos

Vea el siguiente video [BBC - The Code - The Wisdom of the Crowd](https://www.youtube.com/watch?v=iOucwX7Z1HU)    ( 5 min)


Lea los siguientes artículos


*   [The Wisdom of Crowds (Vox Populi) by Francis Galton](https://www.all-about-psychology.com/the-wisdom-of-crowds.html)  (10 min)
*   [A Gentle Introduction to Ensemble Learning](https://machinelearningmastery.com/what-is-ensemble-learning/)  (10 min)





---



##  2.  Zero2Hero   primera parte
Se han lanzado los primeros fascículos coleccionables llamados "from Zero to Hero" que muy detalladamente, paso a paso enseñan todo lo necesario de R para entender los scripts oficiales de la asignatura.
Están en el repositorio oficial de la asignatura, carpeta  **src/zero2hero**



---



## 3.  Grid Search

Busque en internet el preciso significado de los hiperparámetros de la librería **rpart**  que está implementando el algoritmo **CART**  Classification and Regression Trees  propuesto en el año 1984 por Leo Breiman:

*   cp
*   maxdepth
*   minsplit
*   minbucket

Entienda que valores podría tomar razonablemente cada hiperparámetro,  en particular profundice en el hiperparámetro  **cp**  y en la posibilidad de que tome valores negativos.  Es válido consultar a su amigo de *capacidades especiales*  Claude / ChatGPT


En las siguientes celdas encontrará un notebook incompleto, un esqueleto de código brindado a modo de facilitarle la tarea de codeo y permitir que su valiosa cognición se concentre en temas conceptuales de Ciencia de Datos.

Modifíquelo agregando loops para que recorra LOS 4 hiperparámetros de rpart  < cp, maxdepth, minsplit, minbucket >, y luego póngalo a correr. Recuerde cambiar por SU semilla
Tenga muy presente la granularidad que eligirá para cada hiperparámetro.

## Generacion de la clase_ternaria

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

In [1]:
require( "data.table" )

# rutas: funciona tanto desde la carpeta principal como desde "Clase 2"
carpeta <- if (dir.exists("Primera competencia")) {
  "Primera competencia"
} else {
  "../Primera competencia"
}

# leo el dataset
dataset <- fread(file.path(carpeta, "competencia_01_crudo.csv"))

# calculo el periodo0 consecutivo
dsimple <- dataset[, list("pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]

# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1

# calculo los leads de orden 1 y 2
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]

# pego el resultado en el dataset original y grabo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

fwrite( dataset,
    file = file.path(carpeta, "competencia_01.csv.gz"),
    sep = ","
)

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




limpio el ambiente de R

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,732747,39.2,1492885,79.8,NA,1492885,79.8
Vcells,1374055,10.5,170869263,1303.7,16384,212624218,1622.2


In [3]:
options(repos = c(CRAN = "https://cloud.r-project.org"))

# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")

if(!require("R.utils")) install.packages("R.utils")
require("R.utils")

if (!require("primes")) install.packages("primes")
require("primes")

Loading required package: rpart

Loading required package: parallel

Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings


Loading required package: primes



Aqui debe poner SU semiila primigenia

In [4]:
PARAM <- list()
# reemplazar por su primer semilla
PARAM$semilla_primigenia <- 333331
PARAM$qsemillas <- 5

PARAM$training_pct <- 70L  # entre  1L y 99L

In [5]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30), agrupa=clase_ternaria, seed=semilla)
#   crea una particion 70, 30

particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [6]:
ArbolEstimarGanancia <- function(semilla, training_pct, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(training_pct, 100L -training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_basicos
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 1072500, -27500),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(
    c( list("semilla" = semilla),
      param_basicos,
      list( "ganancia_test" = ganancia_test_normalizada )
     )
  )
}


In [7]:
ArbolesMontecarlo <- function(semillas, param_basicos) {

  # la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
  #  tantas veces como valores tenga el vector  PARAM$semillas
  salida <- mcmapply(ArbolEstimarGanancia,
    semillas, # paso el vector de semillas
    MoreArgs = list(PARAM$training_pct, param_basicos), # aqui paso el segundo parametro
    SIMPLIFY = FALSE,
    mc.cores = detectCores()
  )

  return(salida)
}


In [8]:
# carpeta de trabajo
# por favor cambiar numero de experimento si se cambia el loop principal
carpeta <- normalizePath(
  if (dir.exists("Primera competencia")) "Primera competencia" else "../Primera competencia",
  mustWork = TRUE
)

experimento <- "HT2900"
dir.create(file.path(carpeta, "exp", experimento), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(carpeta, "exp", experimento))

In [9]:
# lectura del dataset
dataset <- fread(file.path(carpeta, "competencia_01.csv.gz"))

# trabajo solo con los datos de 202106, ultimo mes con clase_ternaria completa
# filtro datos
dataset <- dataset[foto_mes==202106]

invisible(gc(full=TRUE, verbose=FALSE)) # garbage collection

nrow(dataset)
dataset[, .N, clase_ternaria]

[1] 164114

clase_ternaria,N
<chr>,<int>
CONTINUA,162142
BAJA+2,1098
BAJA+1,874


In [10]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )


In [11]:
# genero la data.table donde van los resultados detallados del Grid Search
# un registro para cada combinacion de < semilla, parametros >

if(file.exists("gridsearch_detalle.txt")){
  tb_grid_search_detalle <- fread("gridsearch_detalle.txt")
}else{
  tb_grid_search_detalle <- data.table(
    semilla= integer(),
    cp= numeric(),
    maxdepth= integer(),
    minsplit= integer(),
    minbucket= integer(),
    ganancia_test= numeric()
  )
}

nrow( tb_grid_search_detalle )

[1] 0

Esta es la parte del código que usted debe expandir a TODOS los hiperparámetros de rpart,
<br>ya que actualmente apenas recorre  maxdepth y  minsplit  dejando fijos  cp=-0.5  y minbucket=5

In [12]:
# calibración: tiempo de UNA combinación con TODAS tus semillas actuales
t0 <- Sys.time()
prueba <- ArbolesMontecarlo(
  PARAM$semillas,
  list(cp = -1, maxdepth = 14L, minsplit = 20L, minbucket = 5L)  # combinación "cara"
)
t1 <- Sys.time()
tiempo_por_combinacion <- as.numeric(difftime(t1, t0, units = "secs"))

cat("Tiempo combinación cara:", round(tiempo_por_combinacion, 1), "seg\n")
cat("Núcleos detectados:", detectCores(), "\n")
cat("Nota: la estimación del total (tiempo x cantidad de combinaciones)\n",
    "se ve mejor una vez corrida la celda del loop grande, que ya la calcula\n",
    "e imprime el ETA en vivo.\n")

Tiempo combinación cara: 60.2 seg
Núcleos detectados: 8 
Nota: la estimación del total (tiempo x cantidad de combinaciones)
 se ve mejor una vez corrida la celda del loop grande, que ya la calcula
 e imprime el ETA en vivo.


In [ ]:
# Grilla inicial moderada
valores_cp <- c(
  -1,
  0,
  1e-4,
  1e-3,
  1e-2
)

valores_maxdepth <- c(
  4L, 6L, 8L, 10L, 12L, 14L
)

valores_minsplit <- c(
  1000L, 400L, 100L, 20L
)

valores_minbucket <- c(
  5L, 10L, 20L, 50L
)

archivo_grid <- "gridsearch_detalle.txt"

# Cantidad de combinaciones válidas
q_combinaciones <- length(valores_cp) *
  length(valores_maxdepth) *
  sum(vapply(
    valores_minsplit,
    function(x) sum(2L * valores_minbucket <= x),
    integer(1)
  ))

cat(
  "Combinaciones:", q_combinaciones, "\n",
  "Árboles:", q_combinaciones * length(PARAM$semillas), "\n"
)

iter <- 0L
t_inicio <- Sys.time()

for (vcp in valores_cp) {

  for (vmax_depth in valores_maxdepth) {

    for (vmin_split in valores_minsplit) {

      for (vmin_bucket in valores_minbucket) {

        # Cada split debe poder producir dos hojas con al menos
        # minbucket observaciones cada una
        if (2L * vmin_bucket > vmin_split) {
          next
        }

        iter <- iter + 1L

        cat(sprintf(
          "%d/%d | cp=%g | depth=%d | split=%d | bucket=%d\n",
          iter,
          q_combinaciones,
          vcp,
          vmax_depth,
          vmin_split,
          vmin_bucket
        ))

        flush.console()

        # Semillas ya ejecutadas para esta combinación exacta
        semillas_realizadas <- tb_grid_search_detalle[
          abs(cp - vcp) < 1e-12 &
            maxdepth == vmax_depth &
            minsplit == vmin_split &
            minbucket == vmin_bucket,
          unique(semilla)
        ]

        # Permite continuar si la ejecución fue interrumpida
        semillas_pendientes <- setdiff(
          PARAM$semillas,
          semillas_realizadas
        )

        if (length(semillas_pendientes) == 0L) {
          cat("  Ya calculada; se omite.\n")
          next
        }

        param_basicos <- list(
          cp = vcp,
          maxdepth = vmax_depth,
          minsplit = vmin_split,
          minbucket = vmin_bucket
        )

        ganancias <- ArbolesMontecarlo(
          semillas_pendientes,
          param_basicos
        )

        nuevos_resultados <- rbindlist(
          ganancias,
          fill = TRUE
        )

        tb_grid_search_detalle <- rbindlist(
          list(
            tb_grid_search_detalle,
            nuevos_resultados
          ),
          use.names = TRUE,
          fill = TRUE
        )

        # Guardar después de cada combinación
        fwrite(
          tb_grid_search_detalle,
          file = archivo_grid,
          sep = "\t"
        )
      }
    }
  }
}


In [ ]:
# ============================================================
# HORA DE FINALIZACION DEL GRID SEARCH
# ============================================================
cat("Inicio: ", format(t_inicio, "%Y-%m-%d %H:%M:%S"), "\n")
cat("Fin:    ", format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "\n")
cat("Duracion total:", round(as.numeric(difftime(Sys.time(), t_inicio, units = "hours")), 2), "horas\n")

In [14]:
# cantidad de registros de la tabla
nrow(tb_grid_search_detalle)

[1] 2100

In [15]:
# muestro la tabla
tb_grid_search_detalle

semilla,cp,maxdepth,minsplit,minbucket,ganancia_test
<int>,<dbl>,<int>,<int>,<int>,<dbl>
957877,-1,4,1000,5,311208333
592139,-1,4,1000,5,284625000
263273,-1,4,1000,5,345216667
742057,-1,4,1000,5,307541667
119389,-1,4,1000,5,353191667
957877,-1,4,1000,10,311208333
592139,-1,4,1000,10,284625000
263273,-1,4,1000,10,345216667
742057,-1,4,1000,10,307908333


In [16]:
# genero y grabo el resumen
tb_grid_search <- tb_grid_search_detalle[,
  list( "ganancia_mean" = mean(ganancia_test),
    "qty" = .N ),
  list( cp, maxdepth, minsplit, minbucket )
]


In [17]:
# ordeno descendente por ganancia
setorder( tb_grid_search, -ganancia_mean )


In [18]:
# veo los 10 mejores hiperparámetros
tb_grid_search[1:10]

cp,maxdepth,minsplit,minbucket,ganancia_mean,qty
<dbl>,<int>,<int>,<int>,<dbl>,<int>
-1,6,400,10,333813333,5
-1,6,1000,10,332530000,5
-1,6,100,10,331906667,5
-1,6,20,10,330733333,5
-1,6,1000,5,330073333,5
-1,6,400,5,328808333,5
-1,6,400,20,327598333,5
-1,8,1000,5,327360000,5
-1,6,100,5,326663333,5


In [19]:
# genero un id a la tabla
tb_grid_search[, id := .I ]

fwrite( tb_grid_search,
  file= "gridsearch.txt",
  sep= "\t"
)


# 4.  Análisis de resultados de Grid Search

La salida de la corrida anterior queda en  exp/HT2900  que corresponde a su Google Drive
<br>HT significa Hyperparameter Tuning
<br>El Grid Search es un método de fuerza bruta de un altísimo costo computacional.
<br>Queremos ver si es posible crear un algoritmo de optimización de hiperparámetros que se ahorre recorrer ciertas porciones muy malas del espacio de búsqueda. Algo del estilo “cada vez que pruebo una combinación de hiperparámetros donde  cp > 1 , la ganancia es muy mala, con lo cual ni vale la pena perder el tiempo explorando en esa región”


<br>Levante el archivo de salida gridsearch.txt  a una planilla tipo Excel y analícelo detenidamente
<br>Ordene por ganancia_mean descendente, de la mejor a la peor
<br>En la Planilla Colaborativa, hoja "Grid Search"  cargue  la posición 1, 2, 5, 10, 50 y 100 de su salida
<br>
<br>El de mayor ganancia_mean  decimos que es el primero del ranking
En el channel  #J-Tarea Hogar 02 , topic Analisis Grid Search   intente contestar estas preguntas:

* ¿Qué combinaciones de hiperparámetros poseen una ganancia muy buena?
* ¿Hay algún hiperparámetro que para cierto valor siempre genera una ganancia muy mala, independientemente de lo que valgan los otros hiperparámetros?
* ¿Qué combinaciones de hiperparámetros es pésima y hubiera sido bueno ahorrarse esas corridas?

(tiempo estimado 30 minutos, dificultad media )
